In [ ]:
# Note: inside Jupyter we write %pip, not plain pip.
%pip install ugot

In [ ]:
import time
from ugot import ugot
got = ugot.UGOT()
got.initialize('192.168.1.186')     # <-- change this to YOUR robot's number!
print("UGOT connected.")
got.play_audio_tts("Hello World!", 0, False)
got.show_light_rgb_effect(255,255,255,2)
time.sleep(2)
got.turn_off_lights()

# Level 3 term 5

## Session 1

1. what is a tuple

In [ ]:
# A tuple is a group of values that belong together.
# We write a tuple with round brackets ( ).

my_colour = (255, 0, 0)
print(my_colour)
print(type(my_colour))
print(len(my_colour))

In [ ]:
# You already know the list. A list uses square brackets [ ].
my_list = [255, 0, 0]
print(my_list)
print(type(my_list))

2. tuples are locked

In [ ]:
# A LIST can be changed.
my_list = [255, 0, 0]
my_list[0] = 100
print(my_list)

In [ ]:
# A TUPLE cannot be changed. Run this cell and read the error message!
my_colour = (255, 0, 0)
my_colour[0] = 100

The error says:

`TypeError: 'tuple' object does not support item assignment`

A tuple is **locked**. That is a good thing! When the robot tells you
"the face is 72 pixels wide", that is a **fact** about what the camera saw.
You should not be able to change a fact by accident.

Use a **list** when the values will change (your Lucky 21 hand grows every turn).
Use a **tuple** when the values belong together and should stay put (a colour, a position).

3. unpacking a tuple

In [ ]:
my_colour = (255, 0, 0)

# The long way. You have written code like this before:
red = my_colour[0]
green = my_colour[1]
blue = my_colour[2]
print(red, green, blue)

In [ ]:
# The short way. This is called UNPACKING.
red, green, blue = my_colour
print(red, green, blue)

In [ ]:
# Unpacking works on any tuple, as long as you have the right number of boxes.
student = ("Jack", 12, "Singapore")
name, age, country = student
print(f"{name} is {age} years old and lives in {country}.")

In [ ]:
# What if you get the number of boxes wrong? Run it and read the error.
student = ("Jack", 12, "Singapore")
name, age = student

`ValueError: too many values to unpack (expected 2)`

Python is telling you exactly what went wrong: it had 3 values but only 2 boxes.
Reading the error message is a superpower. It almost always tells you the answer.

4. a tuple of one

In [ ]:
# Careful! Round brackets on their own are NOT a tuple.
not_a_tuple = (5)
print(type(not_a_tuple))

# A tuple of one value needs a comma.
really_a_tuple = (5,)
print(type(really_a_tuple))

## Session 2

1. what's in the box?

Surprise: **the robot has been giving you tuples all term.**
Every time you called a sensor, it handed you a group of values.
Let's look inside the box.

In [ ]:
# ROBOT NEEDED. Put the robot on a black line first.
got.load_models(["line_recognition"])
got.set_track_recognition_line(0)

line_info = got.get_single_track_total_info()
print(line_info)
print(type(line_info))
print(len(line_info))

In [ ]:
# ROBOT NEEDED. Look at the camera while you run this.
got.load_models(["face_recognition"])

face_info = got.get_face_recognition_total_info()
print(face_info)
print(len(face_info))

if face_info:
    print(face_info[0])
    print(len(face_info[0]))

Before you write any more code, always ask the robot three questions:

1. `print(...)`  — what did you send me?
2. `type(...)`   — what kind of thing is it?
3. `len(...)`    — how many values are inside?

Measure first, then index. This one habit prevents most sensor bugs.

2. line follow, the tuple way

In [ ]:
# The OLD way, from Term 4:
line_info = got.get_single_track_total_info()
offset = line_info[0]
line_type = line_info[1]
print(offset, line_type)

In [ ]:
# The TUPLE way. Three lines become two!
line_info = got.get_single_track_total_info()
offset, line_type = line_info
print(offset, line_type)

If that cell says `too many values to unpack`, it means the robot sends
**more** than two values. No problem — take the two you want:

```python
offset, line_type = line_info[0], line_info[1]
```

Use `len(line_info)` from Session 2 to find out how many there really are.

In [ ]:
# The whole line follower, using unpacking.
import time
got.load_models(["line_recognition"])
got.set_track_recognition_line(0)

while True:
    offset, line_type = got.get_single_track_total_info()

    if line_type > 0:
        got.mecanum_move_xyz(0, 25, int(0.25 * offset))
        time.sleep(0.1)
    else:
        got.mecanum_stop()
        break

print("There's no line to follow, let's stop!")

3. a list of tuples

In [ ]:
# Each step of the maze is a tuple: (what to do, how much).
# All the steps together are a list.

route = [
    ("forward", 20),
    ("left", 90),
    ("forward", 30),
    ("right", 90),
    ("forward", 20),
]

print(route)
print(route[0])
print(route[0][0])

In [ ]:
# Unpacking inside a for loop. This is the big one!
for action, amount in route:
    print(f"Now I will go {action} by {amount}.")

In [ ]:
# ROBOT NEEDED. The same route, but the robot drives it.
import time

for action, amount in route:
    if action == "forward":
        got.mecanum_move_speed_times(0, 20, amount, 1)
    elif action == "left":
        got.mecanum_turn_speed_times(2, 30, amount, 2)
    elif action == "right":
        got.mecanum_turn_speed_times(3, 30, amount, 2)
    time.sleep(0.5)

got.mecanum_stop()
print("Route finished!")

**Think about it:** the maze changed shape. How many lines of your
*instructions* do you have to rewrite?

None! You only change the `route` data. That is why programmers love
lists of tuples.

## Session 3

1. the problem with mystery numbers

Here is a line from your Term 4 face tracking code:

```python
face_width = face_info[0][4]
```

Quick — **what does the `4` mean?** And what was `1` again?

Nobody can remember. And if you type `3` by mistake, Python will not
complain. Your robot will just behave strangely and you will not know why.

A **dictionary** solves this by giving every value a **name**.

2. making a dictionary

In [ ]:
# A dictionary uses curly brackets { }.
# Every item is a KEY (a name) and a VALUE, joined by a colon.

student = {"name": "Jack", "age": 12, "country": "Singapore"}

print(student)
print(type(student))
print(len(student))

In [ ]:
# You look things up by NAME, not by number.
print(student["name"])
print(student["age"])

print(f"{student['name']} is {student['age']} years old.")

3. changing and adding

In [ ]:
# Unlike a tuple, a dictionary CAN be changed.
student["age"] = 13
print(student)

# Adding a brand new key is just as easy.
student["favourite_food"] = "beef noodle"
print(student)

4. looking up safely

In [ ]:
# What if the key is not there? Run it and read the error.
print(student["height"])

In [ ]:
# Way 1: check first with 'in'.
if "height" in student:
    print(student["height"])
else:
    print("I don't know that student's height.")

In [ ]:
# Way 2: use .get() and give a backup answer.
print(student.get("height", "unknown"))
print(student.get("name", "unknown"))

5. looping through a dictionary

In [ ]:
print(student.keys())
print(student.values())

In [ ]:
# .items() gives you a tuple for each item, so you can unpack it!
for key, value in student.items():
    print(f"{key} --> {value}")

## Session 4

1. colour palette

Now we put both new tools together: a **dictionary** whose **values are tuples**.

In [ ]:
COLOURS = {
    "red":    (255, 0, 0),
    "green":  (0, 255, 0),
    "blue":   (0, 0, 255),
    "yellow": (255, 255, 0),
    "purple": (160, 0, 255),
    "white":  (255, 255, 255),
}

print(COLOURS["red"])
print(COLOURS["purple"])

In [ ]:
# Look up the colour, then unpack the tuple.
red, green, blue = COLOURS["yellow"]
print(red, green, blue)

In [ ]:
# ROBOT NEEDED. Ask for a colour and light it up.
wanted = input("What colour do you want? ").lower()

if wanted in COLOURS:
    red, green, blue = COLOURS[wanted]
    got.show_light_rgb_effect(red, green, blue, 2)
    got.play_audio_tts(f"Here is {wanted}", 0, False)
    time.sleep(3)
    got.turn_off_lights()
else:
    print("I don't know that colour yet!")
    got.play_audio_tts("I do not know that colour", 0, False)

In [ ]:
# ROBOT NEEDED. A rainbow show, using .items().
import time

for name, colour in COLOURS.items():
    red, green, blue = colour
    print(f"Now showing {name}.")
    got.show_light_rgb_effect(red, green, blue, 2)
    time.sleep(1.5)

got.turn_off_lights()

2. robot remote control

Remember writing long chains of `if` ... `elif` ... `elif` ...?
A dictionary can replace the whole thing.

In [ ]:
# Each key is a keyboard letter. Each value is an (x, y, z) tuple.
MOVES = {
    "w": (0, 25, 0),      # forward
    "s": (0, -25, 0),     # backward
    "a": (-25, 0, 0),     # slide left
    "d": (25, 0, 0),      # slide right
    "q": (0, 0, 25),      # spin left
    "e": (0, 0, -25),     # spin right
}

for key, move in MOVES.items():
    print(f"Press {key} to move {move}")

In [ ]:
# ROBOT NEEDED. Drive the robot with your keyboard!
import time

while True:
    key = input("Drive me (w/a/s/d/q/e), or x to quit: ").lower()

    if key == "x":
        print("Bye!")
        break

    if key in MOVES:
        x, y, z = MOVES[key]
        got.mecanum_move_xyz(x, y, z)
        time.sleep(0.5)
        got.mecanum_stop()
    else:
        print("I don't know that key!")

**Think about it:** you want to add a new move. With `if` / `elif` you
would add 3 more lines of instructions. With a dictionary you add **one
line of data**. Adding to `MOVES` is all it takes.

## Session 5

1. giving the robot's eyes some names

Time to fix those mystery numbers for good. We will write a helper
function that takes the confusing sensor tuple and hands back a
friendly dictionary.

In [ ]:
def read_face():
    info = got.get_face_recognition_total_info()

    if not info:
        return None                      # no face in front of the camera

    face_x = info[0][1]                  # how far left or right the face is
    face_width = info[0][4]              # how big the face looks (bigger = closer)

    return {"x": face_x, "width": face_width}

In [ ]:
# ROBOT NEEDED. Now the readings have names!
import time

for i in range(10):
    face = read_face()

    if face is None:
        print("I cannot see anybody.")
    else:
        print(f"Face at x = {face['x']}, width = {face['width']}")

    time.sleep(1)

2. a settings dictionary

Your old code had numbers like `70`, `300` and `340` sprinkled everywhere.
When you wanted to change one, you had to hunt through every cell.

Put them all in one dictionary instead.

In [ ]:
SETTINGS = {
    "speed": 25,          # how fast we drive
    "close_enough": 70,   # face width that means "stop, you are close enough"
    "center_min": 300,    # left edge of the "middle of the picture"
    "center_max": 340,    # right edge of the "middle of the picture"
}

print(SETTINGS["speed"])
print(SETTINGS["close_enough"])

3. the face approacher, rebuilt

In [ ]:
# ROBOT NEEDED. Drive forward until the face is close enough.
import time

while True:
    face = read_face()

    if face is None:
        # No face! Keep looking, but do not charge ahead blindly.
        got.mecanum_move_xyz(0, 15, 0)

    elif face["width"] > SETTINGS["close_enough"]:
        got.mecanum_stop()
        print("Close enough. Hello!")
        got.play_audio_tts("Hello, nice to meet you!", 0, False)
        break

    else:
        got.mecanum_move_xyz(0, SETTINGS["speed"], 0)

    time.sleep(0.1)

Compare this with the version you wrote in Term 4:

* `face["width"]` says what it means. `face_info[0][4]` does not.
* `if face is None` means the robot has a plan when nobody is there.
* Every number lives in `SETTINGS`, so changing one changes the whole robot.

4. can you find the bug?

Here is a face tracker with a real bug in it. It stops when the face is
between **300** and **350**, but it only steers when the face is below
**300** or above **380**.

So what happens to a face sitting at **x = 360**?

Read it carefully, then talk to your partner before you run it.

In [ ]:
# BUGGY! Read it, do not trust it.
import time

while True:
    face = read_face()

    if face:
        if face["width"] > 70 and 300 < face["x"] < 350:
            got.mecanum_stop()
            print("Let's stop")
            break
        elif face["x"] > 380:
            got.mecanum_move_xyz(20, 10, 0)
        elif face["x"] < 300:
            got.mecanum_move_xyz(-20, 10, 0)
        else:
            got.mecanum_move_xyz(0, 10, 0)
    else:
        got.mecanum_move_xyz(0, 10, 0)

    time.sleep(0.1)

**The answer:** a face at x = 360 is not below 300 and not above 380, so
the robot decides it is "in the middle" and drives straight at it. But
360 is not between 300 and 350, so the stopping rule never becomes true.
The robot never stops.

**Your job:** rewrite it using `SETTINGS` so the stopping zone and the
steering zone use the *same* numbers. Do it in the empty cell below.

5. the diagonal approach

In [ ]:
# ROBOT NEEDED. Steer and approach at the same time.
def face_diagonal_centralize():
    import time

    while True:
        face = read_face()

        if face is None:
            got.mecanum_move_xyz(0, 10, 0)          # no face, search forward

        elif (face["width"] > SETTINGS["close_enough"]
              and SETTINGS["center_min"] < face["x"] < SETTINGS["center_max"]):
            got.mecanum_stop()
            print("Face centered!")
            break

        elif face["x"] < SETTINGS["center_min"]:
            got.mecanum_move_xyz(-20, 10, 0)        # face is left, go left + forward

        elif face["x"] > SETTINGS["center_max"]:
            got.mecanum_move_xyz(20, 10, 0)         # face is right, go right + forward

        else:
            got.mecanum_move_xyz(0, 10, 0)          # centered but still far away

        time.sleep(0.1)

In [ ]:
face_diagonal_centralize()

## Session 6

1. lucky 21 with real cards

Your Lucky 21 game used `random.randint(1, 10)`. But a real deck has a
Jack, a Queen, a King and an Ace!

A dictionary can hold the **name** of each card and how many **points**
it is worth.

In [ ]:
CARD_VALUES = {
    "2": 2, "3": 3, "4": 4, "5": 5, "6": 6,
    "7": 7, "8": 8, "9": 9, "10": 10,
    "J": 10, "Q": 10, "K": 10, "A": 11,
}

print(CARD_VALUES["K"])
print(CARD_VALUES["A"])
print(list(CARD_VALUES.keys()))

In [ ]:
import random

# Draw one random card NAME from the dictionary.
card = random.choice(list(CARD_VALUES))
print(f"You drew: {card}")
print(f"It is worth {CARD_VALUES[card]} points.")

In [ ]:
def hand_total(hand):
    total = 0
    for card in hand:
        total = total + CARD_VALUES[card]
    return total


my_hand = ["K", "7", "A"]
print(my_hand)
print(hand_total(my_hand))

In [ ]:
import random

print("Welcome to LUCKY 21 with real cards!")

player_hand = []
dealer_hand = []

# give two cards to the player and the dealer
for i in range(2):
    player_hand.append(random.choice(list(CARD_VALUES)))
    dealer_hand.append(random.choice(list(CARD_VALUES)))

print(f"Your cards : {player_hand}")
print(f"Your total : {hand_total(player_hand)}")

while hand_total(player_hand) < 21:
    choice = input("Hit or stand? Enter h or s: ").lower()

    if choice == "h":
        new_card = random.choice(list(CARD_VALUES))
        player_hand.append(new_card)
        print(f"You receive : {new_card}")
        print(f"Your cards  : {player_hand}")
        print(f"Your total  : {hand_total(player_hand)}")

    elif choice == "s":
        print("You chose to stand.")
        break

    else:
        print("Please enter h or s.")

# the dealer plays
player_total = hand_total(player_hand)

if player_total > 21:
    print("You went over 21, you lose.")

else:
    while hand_total(dealer_hand) < 17:
        dealer_hand.append(random.choice(list(CARD_VALUES)))

    dealer_total = hand_total(dealer_hand)

    print("FINAL RESULT!")
    print(f"Your cards   : {player_hand}  total {player_total}")
    print(f"Dealer cards : {dealer_hand}  total {dealer_total}")

    if dealer_total > 21:
        print("The dealer went over 21. You win!")
    elif player_total > dealer_total:
        print("The player wins!")
    elif player_total < dealer_total:
        print("The dealer wins!")
    else:
        print("It is a draw!")

**Challenge for the fast finishers:** an Ace is worth 11 points, but if
that would take you over 21 it should only be worth 1. Can you change
`hand_total()` so it does that?

2. the robot knows you

The robot already learned your face with
`got.face_recognition_add_name("Ryan")`.

Now let's give it a dictionary so it knows something *about* you too.
Every student should add their own line!

In [ ]:
# Each value is a tuple: (what to say, what colour to show)
STUDENTS = {
    "Ryan": ("Hello Ryan, ready for robotics?", (0, 0, 255)),
    "Jack": ("Good evening Jack!",              (0, 255, 0)),
    # <-- add your own name here!
}

for name, details in STUDENTS.items():
    greeting, colour = details
    print(f"{name}: says '{greeting}' and shows {colour}")

In [ ]:
def greet(name):
    if name in STUDENTS:
        greeting, colour = STUDENTS[name]
        red, green, blue = colour
        got.show_light_rgb_effect(red, green, blue, 2)
        got.play_audio_tts(greeting, 0, False)
        time.sleep(3)
        got.turn_off_lights()
    else:
        got.play_audio_tts("I do not know you yet!", 0, False)

In [ ]:
# ROBOT NEEDED.
greet("Ryan")

In [ ]:
# ROBOT NEEDED. Walk up to the robot and let it greet you.
import time

got.load_models(["face_recognition"])

while True:
    face = read_face()

    if face is None:
        got.mecanum_move_xyz(0, 15, 0)
    elif face["width"] > SETTINGS["close_enough"]:
        got.mecanum_stop()
        greet("Ryan")            # <-- try your own name
        break
    else:
        got.mecanum_move_xyz(0, SETTINGS["speed"], 0)

    time.sleep(0.1)

## Session 7

1. tuple or dictionary or list?

| I want to store...                          | Use a...    | Looks like             |
|---------------------------------------------|-------------|------------------------|
| My Lucky 21 hand, which grows every turn     | list        | `["K", "7"]`           |
| One colour: red, green and blue together     | tuple       | `(255, 0, 0)`          |
| A sensor reading I should not change         | tuple       | `(120, 1)`             |
| Things I look up by name                     | dictionary  | `{"speed": 25}`        |
| All my robot's settings in one place         | dictionary  | `SETTINGS`             |
| Many colours, each with a name               | dictionary of tuples | `COLOURS`     |

Round brackets `( )` = tuple, locked.
Square brackets `[ ]` = list, changeable.
Curly brackets `{ }` = dictionary, look up by name.

2. quick quiz

In [ ]:
# What will each line print? Write your guess FIRST, then run it.
things = {"a": (1, 2), "b": [3, 4]}

print(len(things))
print(things["a"])
print(things["a"][1])
print(type(things["b"]))
print("c" in things)
print(things.get("c", "not here"))

## practice